# `Function3D` — Working with 3D Scalar Fields

This notebook introduces `Function3D`, the core scalar-field object in **pygeoinf3D**.
A `Function3D` wraps a vectorised callable over a `BoxDomain` and supports:

- Pointwise arithmetic (`+`, `-`, `*` scalar, `pointwise_mul`)
- Evaluation at individual points and on grids
- Volumetric integration via `BoxDomain.integrate_volume`
- Static axis-aligned 2-D slice visualisation via `plot_function3d_axis_slice`
- Optional interactive slice sliders via `plot_function3d_axis_slider`
- Space attachment / detachment for use inside `Lebesgue3D`

**Notebook series:**  
`1_box_domain_demo.ipynb` → **`2_functions_3d_demo.ipynb`** → `3_lebesgue_space_3d_demo.ipynb`

---
> **Key API note:** the `evaluate_callable` must accept a single `np.ndarray` of shape `(N, 3)` (or `(3,)`)  
> and return an array of shape `(N,)` (or a scalar). Index components as `x[..., 0]`, `x[..., 1]`, `x[..., 2]`.

---
> **Visualisation mode:** later in the notebook you can choose `"static"`, `"interactive"`, or `"both"` and re-run the plotting section.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pygeoinf3d import BoxDomain, Function3D
from pygeoinf3d.plot import plot_function3d_axis_slice

# Unit cube Ω = [0,1]³
box = BoxDomain(np.zeros(3), np.ones(3))
print(f"Domain: {box}")

## 1 — Creating `Function3D`

Three representative scalar fields on $\Omega = [0,1]^3$:

| Symbol | Formula | Character |
|--------|---------|----------|
| $f$ | $\sin(\pi x)\cos(\pi y)$ | Oscillatory, $z$-independent |
| $g$ | $x^2 + y^2 + z^2$ | Radial polynomial |
| $h$ | $\exp\!\bigl(-10\,\|\mathbf{x}-\mathbf{c}\|^2\bigr)$ | Gaussian bump at centre $\mathbf{c}=(0.5,0.5,0.5)$ |

The `evaluate_callable` receives a single `np.ndarray` with last dimension 3 and components indexed as `x[..., 0]`, `x[..., 1]`, `x[..., 2]`.

In [ ]:
def _oscillatory(x: np.ndarray) -> np.ndarray:
    """f(x,y,z) = sin(pi*x) * cos(pi*y)  — z-independent oscillation."""
    x = np.asarray(x)
    return np.sin(np.pi * x[..., 0]) * np.cos(np.pi * x[..., 1])

def _radial(x: np.ndarray) -> np.ndarray:
    """g(x,y,z) = x^2 + y^2 + z^2  — grows from origin."""
    x = np.asarray(x)
    return x[..., 0]**2 + x[..., 1]**2 + x[..., 2]**2

def _gauss(x: np.ndarray) -> np.ndarray:
    """h(x,y,z) = exp(-10 * ||x - 0.5||^2)  — Gaussian bump at cube centre."""
    x = np.asarray(x)
    return np.exp(-10.0 * ((x[..., 0] - 0.5)**2
                          + (x[..., 1] - 0.5)**2
                          + (x[..., 2] - 0.5)**2))

f = Function3D(box, evaluate_callable=_oscillatory, name=r"$f$")
g = Function3D(box, evaluate_callable=_radial,      name=r"$g$")
h = Function3D(box, evaluate_callable=_gauss,       name=r"$h$")

# Inspect
for fn in [f, g, h]:
    print(f"  {fn}  |  type={type(fn).__name__}  |  is_attached={fn.is_attached}")

# Evaluate at the cube centre
pt = np.array([0.5, 0.5, 0.5])
print()
print(f"Evaluation at {pt}:")
print(f"  f(pt) = {f(pt):.6f}   (expected sin(pi/2)*cos(pi/2) = 0)")
print(f"  g(pt) = {g(pt):.6f}   (expected 0.75)")
print(f"  h(pt) = {h(pt):.6f}   (expected 1.0 at centre)")

## 2 — Arithmetic Operations

| Operation | Syntax | Returns |
|-----------|--------|---------|
| Addition  | `f + g` | `Function3D` |
| Subtraction | `f - g` | `Function3D` |
| Scalar multiply | `f * 2.0` | `Function3D` |
| Pointwise product | `f.pointwise_mul(g)` | `Function3D` |

All operators create **new** `Function3D` objects; the originals are unchanged.

In [ ]:
f_plus_g  = f + g
f_minus_g = f - g
f_scaled  = f * 2.0
f_times_g = f.pointwise_mul(g)

print("Derived Function3D objects:")
print(f"  f + g          → {type(f_plus_g).__name__}")
print(f"  f - g          → {type(f_minus_g).__name__}")
print(f"  f * 2.0        → {type(f_scaled).__name__}")
print(f"  f.pointwise_mul(g) → {type(f_times_g).__name__}")

# Verify at a test point
pt_test = np.array([0.3, 0.4, 0.6])
fv, gv = f(pt_test), g(pt_test)

print()
print(f"Point: {pt_test}")
print(f"  f(pt) = {fv:.6f},  g(pt) = {gv:.6f}")
print()
print(f"  (f+g)(pt)              = {f_plus_g(pt_test):.6f}")
print(f"  f(pt) + g(pt)          = {fv + gv:.6f}   ← match")
print()
print(f"  (f-g)(pt)              = {f_minus_g(pt_test):.6f}")
print(f"  f(pt) - g(pt)          = {fv - gv:.6f}   ← match")
print()
print(f"  (f*2.0)(pt)            = {f_scaled(pt_test):.6f}")
print(f"  2.0 * f(pt)            = {2.0 * fv:.6f}   ← match")
print()
print(f"  f.pointwise_mul(g)(pt) = {f_times_g(pt_test):.6f}")
print(f"  f(pt) * g(pt)          = {fv * gv:.6f}   ← match")

# Confirm originals unmodified
assert f(pt_test) == fv, "f was modified — should not happen"
assert g(pt_test) == gv, "g was modified — should not happen"
print()
print("Originals unmodified: ✓")

## 3 — Volumetric Integration

`BoxDomain.integrate_volume` evaluates $\int_\Omega \phi(\mathbf{x})\,\mathrm{d}\mathbf{x}$ by  
tensor-product Riemann summation at a specified resolution.
It accepts a `Function3D` directly (which is callable).

Analytical results:

$$
\int_\Omega g \,\mathrm{d}\mathbf{x}
  = \int_0^1 x^2\,\mathrm{d}x \times 3 = \tfrac{1}{3} \times 3 = 1.0
$$

$$
\int_\Omega f \,\mathrm{d}\mathbf{x}
  = \underbrace{\int_0^1 \sin(\pi x)\,\mathrm{d}x}_{2/\pi}
    \underbrace{\int_0^1 \cos(\pi y)\,\mathrm{d}y}_{0}
    \underbrace{\int_0^1 1\,\mathrm{d}z}_{1}
  = 0.0
$$

In [ ]:
RESOLUTION = 60   # grid points per axis

int_g = box.integrate_volume(g, resolution=RESOLUTION)
int_f = box.integrate_volume(f, resolution=RESOLUTION)
int_h = box.integrate_volume(h, resolution=RESOLUTION)

# Gaussian bump: analytic value on R^3 = (pi/10)^(3/2) ≈ 0.0557;
# on [0,1]^3 most of the mass is inside (centre at 0.5, sigma=0.316)
int_h_analytic_inf = (np.pi / 10.0)**(1.5)   # integral over all of R^3

print(f"∫ g dV = {int_g:.6f}   (analytic: 1.0000, error: {abs(int_g - 1.0):.2e})")
print(f"∫ f dV = {int_f:.2e}   (analytic: 0.0,    error: {abs(int_f):.2e})")
print(f"∫ h dV = {int_h:.6f}   (R³ analytic: {int_h_analytic_inf:.6f})")

import numpy.testing as npt
npt.assert_allclose(int_g, 1.0,  rtol=1e-2, err_msg="∫g should be 1.0")
npt.assert_allclose(int_f, 0.0, atol=1e-2,  err_msg="∫f should be 0.0")
print("\nIntegration assertions passed ✓")

In [ ]:
VISUALIZATION_MODE = "interactive"  # change to "interactive" or "both"
INTERACTIVE_AXIS = "z"
INTERACTIVE_N_SLICES = 31
INTERACTIVE_RESOLUTION = 70
INTERACTIVE_FUNCTION_NAMES = ["f", "g", "h"]

try:
    from pygeoinf3d.plot import plot_function3d_axis_slider
    INTERACTIVE_PLOTTING_AVAILABLE = True
except ImportError:
    plot_function3d_axis_slider = None
    INTERACTIVE_PLOTTING_AVAILABLE = False

valid_modes = {"static", "interactive", "both"}
if VISUALIZATION_MODE not in valid_modes:
    raise ValueError(
        f"VISUALIZATION_MODE must be one of {sorted(valid_modes)}, got {VISUALIZATION_MODE!r}"
    )

print(f"VISUALIZATION_MODE = {VISUALIZATION_MODE!r}")
print(f"Interactive plotting available: {INTERACTIVE_PLOTTING_AVAILABLE}")

## 4 — Visualisation Mode

This notebook supports two plotting styles:

- `"static"`: Matplotlib heatmaps embedded directly in the notebook
- `"interactive"`: Plotly slice sliders from `pygeoinf3d.plot`
- `"both"`: render both versions

The interactive option depends on Plotly. If it is not installed, the notebook will print a short message instead of failing.

## 5 — Visualisation: static or interactive slices

You can choose between:

- `plot_function3d_axis_slice(...)` for static Matplotlib heatmaps
- `plot_function3d_axis_slider(...)` for an interactive Plotly slice slider

The cells below respect `VISUALIZATION_MODE`. Set it to `"static"`, `"interactive"`, or `"both"` in the configuration cell above, then re-run this section.

The static view below uses 3 functions × 3 z-values → 9 panels. The interactive view creates one slider figure per selected function.

In [ ]:
functions = [f, g, h]
func_labels = [
    r"$f = \sin(\pi x)\cos(\pi y)$",
    r"$g = x^2 + y^2 + z^2$",
    r"$h = \exp(-10\|\mathbf{x}-0.5\|^2)$",
]
z_coords = [0.25, 0.50, 0.75]

if VISUALIZATION_MODE in {"static", "both"}:
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))

    for row, (fn, flabel) in enumerate(zip(functions, func_labels)):
        for col, z in enumerate(z_coords):
            ax = axes[row, col]
            _, _, _ = plot_function3d_axis_slice(
                fn,
                axis="z",
                coord=z,
                resolution=60,
                show_plot=False,
                ax=ax,
                title=f"z = {z}",
            )
            if col == 0:
                ax.set_ylabel(flabel, fontsize=10)

    fig.suptitle(
        "Axis-aligned z-slices of three Function3D objects",
        fontsize=13,
        y=1.01,
    )
    plt.tight_layout()
    plt.show()
else:
    print(
        "Skipping static grid because VISUALIZATION_MODE='interactive'. "
        "Set VISUALIZATION_MODE to 'static' or 'both' to render it."
    )

In [ ]:
function_map = {
    "f": (f, r"$f = \sin(\pi x)\cos(\pi y)$"),
    "g": (g, r"$g = x^2 + y^2 + z^2$"),
    "h": (h, r"$h = \exp(-10\|\mathbf{x}-0.5\|^2)$"),
}

selected_names = []
for name in INTERACTIVE_FUNCTION_NAMES:
    if name not in function_map:
        raise ValueError(
            f"Unknown function name {name!r}. Choose from {sorted(function_map)}."
        )
    selected_names.append(name)

if VISUALIZATION_MODE in {"static", "both"}:
    fig2, axes2 = plt.subplots(1, 3, figsize=(12, 4))

    for ax, z in zip(axes2, z_coords):
        _, _, _ = plot_function3d_axis_slice(
            h,
            axis="z",
            coord=z,
            resolution=80,
            show_plot=False,
            ax=ax,
            title=f"Gaussian bump — z = {z}",
        )

    fig2.suptitle(
        r"Gaussian bump $h = \exp(-10\|\mathbf{x}-0.5\|^2)$",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

if VISUALIZATION_MODE in {"interactive", "both"}:
    if not callable(plot_function3d_axis_slider):
        print(
            "Interactive plotting requested, but Plotly is not installed or "
            "plot_function3d_axis_slider is unavailable. "
            "Install pygeoinf3d[interactive] or plotly to enable sliders."
        )
    else:
        print(
            f"Rendering interactive {INTERACTIVE_AXIS}-axis sliders for: "
            f"{', '.join(selected_names)}"
        )
        for name in selected_names:
            fn, label = function_map[name]
            fig_slider = plot_function3d_axis_slider(
                fn,
                axis=INTERACTIVE_AXIS,
                n_slices=INTERACTIVE_N_SLICES,
                resolution=INTERACTIVE_RESOLUTION,
                show_plot=False,
                title=f"{label} — interactive {INTERACTIVE_AXIS}-slice slider",
            )
            fig_slider.show()

## 6 — Space Attachment: `copy`, `detach`, `attach_to_space`

A `Function3D` can be **standalone** (`is_attached=False`) or **attached** to a `Lebesgue3D` space  
(`is_attached=True`). Attachment is required when the function is used as an element of the Hilbert space  
(for example as a prior sample or inversion output).

| Method | Action |
|--------|--------|
| `.copy()` | Deep copy, retains attachment state |
| `.detach()` | Returns standalone version (space reference removed) |
| `.attach_to_space(space)` | Returns version attached to `space` |
| `fn.space` | `None` if standalone, `Lebesgue3D` if attached |

In [ ]:
from pygeoinf3d import Lebesgue3D

space = Lebesgue3D(box)

# Copy
g_copy = g.copy()
print(f"g.is_attached         = {g.is_attached}")
print(f"g_copy.is_attached    = {g_copy.is_attached}   (copy preserves state)")
print()

# Attach to a Lebesgue3D space
g_attached = g.attach_to_space(space)
print(f"g_attached.is_attached = {g_attached.is_attached}")
print(f"g_attached.space       = {type(g_attached.space).__name__}")
print()

# Detach from space
g_detached = g_attached.detach()
print(f"g_detached.is_attached = {g_detached.is_attached}")
print(f"g_detached.space       = {g_detached.space}")
print()

# Values are preserved across all transforms
pt2 = np.array([0.3, 0.6, 0.9])
expected = g(pt2)
assert g_copy(pt2)     == expected, "copy changed value"
assert g_attached(pt2) == expected, "attach changed value"
assert g_detached(pt2) == expected, "detach changed value"
print(f"g(pt), copy, attached, detached all agree at {pt2}: {expected:.6f} ✓")
print()

# Arithmetic on standalone functions always creates new standalone Function3D
arith_result = f + g
print(f"(f + g).is_attached = {arith_result.is_attached}   (arithmetic → standalone)")

## Summary

| Feature | Code |
|---------|------|
| Create from vectorised callable | `Function3D(box, evaluate_callable=fn, name='...')` |
| Callable convention | `fn(x: np.ndarray)` with `x[..., 0/1/2]` |
| Evaluate at a point | `f(np.array([x0, y0, z0]))` |
| Addition | `f + g` |
| Subtraction | `f - g` |
| Scalar multiply | `f * 2.0` |
| Pointwise product | `f.pointwise_mul(g)` |
| Volumetric integral | `box.integrate_volume(f, resolution=60)` |
| 2-D axis slice plot | `plot_function3d_axis_slice(f, axis='z', coord=0.5, resolution=60)` |
| Copy | `f.copy()` |
| Attach to Hilbert space | `f.attach_to_space(space)` |
| Detach from space | `f.detach()` |
| Check attachment | `f.is_attached` |
| Get space | `f.space` (None if standalone) |

**Next:** [`3_lebesgue_space_3d_demo.ipynb`](3_lebesgue_space_3d_demo.ipynb) introduces `Lebesgue3D`,  
the $L^2(\Omega)$ Hilbert space whose elements are attached `Function3D` objects.